# Compare and report verified experiment runs

One results reader replaces repeated ROC, threshold, and accuracy code. The comparison requires matching cohort and recording-level split signatures. Only completed runs are shown.

Install the project first with `pip install -e ".[neural,signal,wavelet,notebooks]"`. Opening this notebook does not start training or download data.


In [ ]:
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
print(f"Project: {ROOT.name}")


## Discover completed real-data runs

Synthetic demonstrations are excluded from this ECG comparison. If you deliberately maintain different cohorts, choose one comparable group instead of pooling the metrics.


In [ ]:
import json
import pandas as pd
from ecg_afib.evaluation import compare_runs

run_paths = []
for metadata_path in sorted((ROOT / "runs").glob("*/run.json")):
    metadata = json.loads(metadata_path.read_text())
    if metadata["status"] == "complete" and not metadata["data_provenance"].get("synthetic", False):
        run_paths.append(metadata_path.parent)

if run_paths:
    results = pd.DataFrame(compare_runs(run_paths))
else:
    results = pd.DataFrame()
    print("No completed ECG runs found. Add real data and run the experiment presets first.")
results


## Record-level discrimination and classification

AUROC uses continuous AFib-oriented scores. The confusion matrix uses the threshold chosen on validation records. Average precision is reported as AP, not as trapezoidal PR-AUC.


In [ ]:
from ecg_afib.plots import plot_roc_curves, plot_confusion_matrix

if run_paths:
    roc_figure, roc_axis = plot_roc_curves(run_paths, partition="test")
    confusion_figure, confusion_axis = plot_confusion_matrix(run_paths[0], partition="test")


## Export a deliberate report

Only export after checking the selected runs and their provenance. Report cohort sizes, preprocessing, seed, recording-level splits, threshold rule, and the exact configuration alongside figures. Do not repeatedly tune against the final test results.


In [ ]:
EXPORT_REPORT = False
if EXPORT_REPORT and run_paths:
    destination = ROOT / "reports"
    destination.mkdir(exist_ok=True)
    results.drop(columns=["confusion_matrix"]).to_csv(destination / "experiment_comparison.csv", index=False)
    roc_figure.savefig(destination / "test_roc.pdf", bbox_inches="tight")
    confusion_figure.savefig(destination / "test_confusion_matrix.pdf", bbox_inches="tight")


Read [METHODS.md](../docs/METHODS.md) and [VALIDATION.md](../docs/VALIDATION.md) before writing a performance claim. Recording-level grouping does not establish subject-level independence when subject identifiers are unavailable.
